# 作品タイトル：［COIN］
氏名：［小林凛太郎］

# My Chatbot｜キャラクターと話そう
誰のために、どんな気分になれるBot？ 一文で書こう。
私のために、成長する子供を見るような気分になれるBot。




## Challenge｜気になる改良を選ぼう（任意）
全部やる必要はありません。

| やってみたいこと | 最初の一歩 |
|---|---|
| 返事を増やす | 好きな気分のリストに、一行ずつ追加する |
| 反応する言葉を増やす | `happy_words` や `tired_words` に一行追加する |
| 意図と違う判定を直す | `goodbye` や `not tired` を試す。どの言葉に反応した？ 条件を一つ改善しよう |
| 会話を覚える | `user = {"name": ""}` に呼び名を保存し、返事に使う |
| 新しい気分をつくる | `responses` と `faces` に同じキーを追加し、条件で選ぶ |

### もう一歩先へ（授業の範囲外・任意）
**動くBot：** 話すときに画像や顔を動かす方法を調べ、実装してみよう。授業では扱いません。できなくても問題ありません。

新しい道具を調べたりAIに相談した場合は、何を変え、どう動作確認したかを一言残そう。

## 自分のBotについて
次の二つを、短く書いてください。

**自分で追加したオリジナルの機能・特徴は？**
［画像の表示（時間が足りなかったので３パターンのみ）そしてAPIに接続しWikipediaでの検索機能の追加］

**これからどんな機能を加え、どんなロボットに育てたい？**
［word2vecを導入し単語の組み合わせを理解させ高度に進化させます。また当初の目的であったトランプゲームもまだ導入できていないのでそちらも追加します。画像の種類も＋１０種以上を想定しているのでまだ全体の１０%も出来ておりません。まだまだ伸びしろがあります。］

## 提出
**10月28日(水) 23:59までにUNIPAへ。**

一番上に**タイトル・氏名**。全Cellを実行し、**会話の出力と上の二つの記述を保存した `.ipynb`**を提出。
自分で画像などを追加した場合は、そのファイルも添える。

Cellの実行が完了してから保存しよう。見た目や発展Challengeは任意。大切なのは、**自分のルールで返し、確かめてあること**。

In [1]:
import datetime  # 時間を扱うための標準機能
import json  # 前回から流用
import os  # 画像ファイルの存在確認を行う道具
import random
from urllib.parse import(
    quote,  # 日本語のキーワードをURL用に変換する道具 AIによるとこれが無かったため検索にヒットできないとのこと。
)
from urllib.request import Request, urlopen
from IPython.display import (
    Image,
    clear_output,
    display,
)  # 追加 画面をクリアする道具・画像を表示する道具


# 辞書（dict）：Botのプロフィールを「名前: 値」の形で入れる
bot = {
    "name": "COIN",  # Botの名前
    "role": "ちょっとした検索、プログラミングの実験台",  # Botの役割
    "likes": ["検索", "勉強", "ゲーム"],  # リスト（list）：好きなもの三つ
}

# 気分ごとの顔。キーは normal / happy / surprised / tired の四つ
faces = {
    "normal": "🤖",
    "happy": "🤖✨",
    "surprised": "🤖❗",
    "tired": "🤖💤",
}

# 1. 判定用の言葉リスト（ここは元のまま）
happy_words = [
    "元気",
    "楽しい",
    "うれしい",
    "嬉しい",
    "最高",
    "happy",
    "good",
    "great",
    "fun",
    "fine",
]
tired_words = [
    "元気じゃない",
    "つかれ",
    "疲れ",
    "眠い",
    "しんど",
    "つらい",
    "tired",
    "sleepy",
    "sad",
    "bad",
    "not good",
    "not happy",
]

# --- 挨拶用リストの切り分け ---
morning_words = ["おはよう", "オハヨウ", "おはよ"]
general_greetings = ["こんにちは", "こんばんは", "やあ", "ハロー", "はじめまして"]

# --- コメント集 ---
morning_comments = [
    "今日も一日がんばろうね！☀️",
    "朝ごはんはしっかり食べた？☕",
    "今日も良い日になるといいね！✨",
]
daytime_comments = [
    "お昼ご飯は何にする？🍱",
    "午後も無理せず自分のペースでね！👍",
    "ちょっと一息ついて休憩も挟もうね ☕",
]
night_comments = [
    "今日も一日お疲れ様！ゆっくり休んでね 🌙",
    "美味しい夕飯食べた？🍲",
    "今日もお仕事や勉強、本当にお疲れ様！✨",
]

# 「おはよう」に添える時間帯別コメント
wake_up_daytime_comments = [
    "お昼頃に起きたんだね！よく眠れた？😊",
    "今日も良い一日になりますように！✨",
    "のんびり起きてリフレッシュできたかな？☕",
]
wake_up_night_comments = [
    "こんな時間に起きたの？お仕事や勉強お疲れ様！🌙",
    "夜起きたんだね！体調には気をつけてね",
    "静かな夜の時間、のんびり過ごそうね ✨",
]


# ガード付き画像表示関数（assetsフォルダ内の画像を読み込む）
def show_mood_image(current_mood):
    # assets フォルダ内の画像パスを指定 (.jpg / .png 両対応)
    jpg_path = os.path.join("assets", f"{current_mood}.jpg")
    png_path = os.path.join("assets", f"{current_mood}.png")

    # .jpg があればそれを使い、無ければ .png を探す
    target_path = (
        jpg_path
        if os.path.exists(jpg_path)
        else (png_path if os.path.exists(png_path) else None)
    )

    if target_path:
        try:
            display(Image(filename=target_path, width=180))
        except Exception:
            pass

# プロフィールを表示する
print(f"{faces['normal']} {bot['name']}｜{bot['role']}")

# 最初の画面表示用の初期値を用意
user_message = "（会話スタート）"
mood = "normal"
reply = f"こんにちは！{bot['name']}だよ。今日の気分はどう？（終わる時は「バイバイ」と入力してね）"

# 追加ここから繰り返しの処理（whileループ）
while True:
    # 追加画面をリセットして最新の1画面だけにする
    clear_output(wait=True)

    # 追加 moodに応じた画像の表示（ファイルがない場合はスルー）
    show_mood_image(mood)

    # 追加見栄えを1つの枠のように綺麗に整える表示
    print("┌──────────────────────────────────────────┐")
    print(f"  👤 あなた：{user_message}")
    print("├──────────────────────────────────────────┤")
    print(f"  {faces.get(mood, '🤖')} {bot['name']}：{reply}")
    print("└──────────────────────────────────────────┘")

    # ユーザーの入力を受け取る
    try:
        user_message = input("💬 メッセージを入力：").strip()
    except KeyboardInterrupt:
        user_message = "バイバイ"

    # 終わらすためのゾーン-----------------------------------------------------------------------------------
    # 追加「バイバイ」などの終了キーワードでループを止める
    end_words = [
        "バイバイ",
        "ばいばい",
        "bye",
        "exit",
        "終了",
        "end",
        "終わり",
        "おわり",
        "やめる",
        "ヤメる",
        "さようなら",
        "サヨウナラ",
        "またね",
        "おつかれ",
        "お疲れ",
        "オツカレ",
        "おやすみ",
        "オヤスミ",
        "ばい",
        "バイ",
    ]

    # 「」『』などのカギカッコや末尾の不要な記号・「あ」等の誤入力を取り除いて判定用文字列を作る
    cleaned_input = (
        user_message.lower()
        .replace("「", "")
        .replace("」", "")
        .replace("『", "")
        .replace("』", "")
        .strip("。！!？?あ ")
    )

    # 追加：1つでも終了キーワードが含まれていればループを止める
    if any(word in cleaned_input for word in end_words):
        clear_output(wait=True)
        show_mood_image("happy")
        print("┌──────────────────────────────────────────┐")
        print(
            f"  {faces['happy']} {bot['name']}：また、何時でも話しかけてくださいね～👋"
        )
        print("└──────────────────────────────────────────┘")
        break  # ループを終了する


    # 数当てゲーム（High & Low）ゾーン ---------------------------------------------------Aiによるオマケを入れてみました。
    # 「ゲーム」「数当て」でゲームを開始
    if "ゲーム" in user_message or "数当て" in user_message:
        target_number = random.randint(1, 100)
        bot["game_target"] = target_number  # Botの記憶（辞書）に正解を保存
        mood = "happy"
        reply = "数当てゲームを始めよう！1から100までの数字を覚えたよ。数字を入力して当ててみてね！（諦める時は「降参」と言ってね）"
        continue

    # ゲームプレイ中の処理（Botが数字を覚えている場合）
    if "game_target" in bot:
        if user_message == "降参":
            ans = bot.pop("game_target")  # ゲーム終了
            mood = "tired"
            reply = f"正解は【{ans}】でした！また挑戦してください！"
            continue
        
        # 入力された文字が数字かどうか判定
        if user_message.isdigit():
            num = int(user_message)
            ans = bot["game_target"]
            
            if num < ans:
                mood = "surprised"
                reply = f"【{num}】ですね！もっと「大きい（High）」数字です！"
            elif num > ans:
                mood = "surprised"
                reply = f"【{num}】ですか？もっと「小さい（Low）」数字です！"
            else:
                bot.pop("game_target")  # 正解したのでゲーム終了
                mood = "happy"
                reply = f"🎉 すごい！正解は【{ans}】でした！大正解！"
            continue
    # 自己紹介のためのゾーン------------------------------------------------------------------------------------
    you_words = [
        "あなた",
        "貴方",
        "貴女",
        "アナタ",
        "君",
        "きみ",
        "キミ",
        "お前",
        "おまえ",
        "オマエ",
        "あんた",
        "アンタ",
        "bot",
        "ボット",
        "ai",
        "AI",
        "Ai",
        "Bot",
        "BOT",
        "You",
        "お宅",
        "おたく",
        "御宅",
        "お主",
        "汝",
        "貴君",
        "貴殿",
        "貴公",
        "貴様",
        "キサマ",
        "きさま",
        "you",
        "ユー",
        "そなた",
        "ソナタ",
        "其方",
        "おぬし",
        "お主",
        "御主",
        "あんた",
        "アンタ",
        "エーアイ",
        bot["name"].lower(),
    ]

    # 2. 質問キーワードの固定リスト
    about_me = [
        "誰",
        "だれ",
        "何者",
        "ダレ",
        "なにもの",
        "何もの",
        "ナニモノ",
        "できる",
        "出来",
        "できま",
        "自己紹介",
        "機能",
        "特技",
        "使い方",
    ]
    about_like = ["好き", "すき", "趣味", "好物", "スキ"]
    about_creator_words = [
        "誰が作った",
        "作った人",
        "作者",
        "製作者",
        "生みの親",
        "誰が作",
        "開発者",
        "誰によって",
        "誰に",
        "誰の",
        "どなたに",
    ]
    about_age_words = [
        "何歳",
        "いくつ",
        "年齢",
        "誕生日",
        "生年月日",
        "いつ生まれた",
        "いつ作られた",
    ]
    about_status_words = [
        "元気",
        "調子",
        "なにしてる",
        "何してる",
        "忙しい",
        "起きてる",
    ]

    # いつ 誰に 作られた これは優先度を上げるために上にもってきています。何故作られたのかは3つ項目下のコードにあります。
    if (
        any(w in user_message for w in you_words)
        and any(w in user_message for w in about_age_words)
        and any(w in user_message for w in about_creator_words)
    ):
        mood = "happy"
        reply = "作者さんは『時間があったら入れるね』と言ったきり未実装なんです…。私は2026年10月25日に生まれました！"
        continue
    elif any(w in user_message for w in you_words) and any(
        w in user_message for w in about_creator_words
    ):
        mood = "happy"
        reply = "作者は「時間あったら入れとく～」っと言ってましたね。今は入ってないですね。これは...つめが甘い。"
        continue

    elif any(word in user_message for word in you_words) and any(
        word in user_message for word in about_me
    ):  # anyでどれか一つにするのはわかっていたのですが動かずAIに聞きました。
        # どれか1つでも （ 単語 ～の中に含まれる ユーザーの入力文 ～を繰り返して 単語を ～から キーワードのリスト ） 逆向きに読む。「キーワードのリストから単語を1つずつ取り出して、その単語がユーザーの入力文に含まれているか調べ、どれか1つでも含まれていればTrueにする」any自体は「どれか1つでも含まれていればTrueにする」という暗記。
        mood = "happy"
        reply = f"私は{bot['role']}の『{bot['name']}』と申します！🤖\n「『〇〇』について教えて」と話しかけてくれたらWikipediaで調べますね！"
        continue

    # ★ 「about_me」単体ブロックは誤爆（「誰が作った」で反応してしまう問題）を防ぐため削除済み

    # 3. 上のどれにも当てはまらず、「you_words」単体だけがある場合 ➔ 呼びかけへの反応！
    elif any(word in user_message for word in you_words):
        mood = "happy"
        reply = "はい！何でしょうか？お気軽にどうぞ！"
        continue

    elif any(w in user_message for w in you_words) and any(
        w in user_message for w in about_like
    ):
        mood = "happy"
        reply = "勉強とゲームが好きです！まだまだ知らない言葉も多いですが、頑張りますね。"
        continue

    # 4. 「あなた」×「年齢・誕生日」★追加
    elif any(w in user_message for w in you_words) and any(
        w in user_message for w in about_age_words
    ):
        mood = "happy"
        reply = "2026/10/25に生まれました！"
        continue

    # 5. 「あなた」×「状態・調子」★追加
    elif any(w in user_message for w in you_words) and any(
        w in user_message for w in about_status_words
    ):
        mood = "happy"
        reply = "とっても元気ですよ！今日も一緒に頑張りましょう！"
        continue

    # --- 会話処理（elif） ---

    # パターン1: 「おはよう」と言われた場合（時間は問わず「おはよう」と返す）
    if any(word in user_message for word in morning_words):
        mood = "happy"
        current_hour = datetime.datetime.now().hour

        # 時間帯によって添えるコメントだけを変える
        if 18 <= current_hour or current_hour < 5:
            # 夜〜深夜に「おはよう」と言われた時
            comment = random.choice(wake_up_night_comments)
            if current_hour < 5:
                mood = "surprised"  # 深夜なら驚き顔
        else:
            # 昼〜朝に「おはよう」と言われた時
            comment = random.choice(wake_up_daytime_comments)

        reply = f"おはようございます！{comment}"
        continue

    # パターン2: それ以外の一般的な挨拶（時間帯で「こんにちは/こんばんは」を自動切り替え）
    elif any(word in user_message for word in general_greetings):
        mood = "happy"
        current_hour = datetime.datetime.now().hour

        if 5 <= current_hour < 11:
            reply = f"おはようございます！{random.choice(morning_comments)}"
        elif 11 <= current_hour < 18:
            reply = f"こんにちは！{random.choice(daytime_comments)}"
        else:
            reply = f"こんばんは！{random.choice(night_comments)}"

        continue

    # 言葉を説明のゾーンーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーー
    # 「」や『』が含まれているかチェックする。「←これがあるのかどうか、("「" in user_messageで、」←これがあるのかどうか、"」" in user_messageでそれをandでつなぐ。
    has_kakko = ("「" in user_message and "」" in user_message) or (
        "『" in user_message and "』" in user_message
    )

    # 検索っぽい雰囲気を察知するためのキーワード
    search_hints = [
        "教え",
        "答え",
        "？",
        "?",
        "何",
        "とは",
    ]  # もともとorでしていましたがAIによると「条件成立（True）」になってしまうバグがあるそうなので変更。
    has_hint = any(hint in user_message for hint in search_hints)

    # パターン①：カギカッコが付いている場合 ➔ 中身を正確に検索できる。
    if has_kakko:
        # 「」または『』の中身を取り出す
        if "「" in user_message and "」" in user_message:
            clean_keyword = (
                user_message.split("「", 1)[1].split("」", 1)[0].strip()
            )  # 「」の中を取り出すのはわかっていたのですが、方法が分からずAIに聞き、まさかここまで複雑そうな内容だとは思っていませんでした。
        else:
            clean_keyword = (
                user_message.split("『", 1)[1].split("』", 1)[0].strip()
            )

        if clean_keyword:
            try:  # 今回の授業中のワークで動かなかったときに調べて見つけたものを使ってみました。
                # 日本語キーワードをURL形式に変換
                encoded_keyword = quote(clean_keyword)
                url = f"https://ja.wikipedia.org/api/rest_v1/page/summary/{encoded_keyword}"

                # 前回の課題の書き方をそのまま使用
                headers = {"User-Agent": "Mozilla/5.0"}
                req = Request(url, headers=headers)

                with urlopen(req, timeout=40) as response:
                    data = json.load(response)

                if "extract" in data:
                    mood = "happy"  # テンプレートをそのまま使用
                    reply = f"【{clean_keyword} についてお調べしました！】\n{data['extract']}"
                else:
                    mood = "normal"
                    reply = f"申し訳ありません、{clean_keyword} についての情報は見つかりませんでした…。"
            except Exception:
                mood = "tired"
                reply = f"申し訳ありません、{clean_keyword} について検索処理中にエラーが発生いたしました。"
        # カッコの中が空っぽ（例: 「『』について教えて」など） ➔ 注意を促す
        else:
            mood = "normal"
            reply = "カッコの中が空っぽのようです！「『Python』について教えて」のように、調べたい言葉を入れてみてくださいね！"
        continue

    # パターン②：カギカッコはないが、検索っぽい言葉が入っている場合 ➔ 入力を誘導する！
    elif has_hint:
        mood = "normal"
        reply = "単語を正確に読み取るため、「『Python』について教えて」のように「」や『』で囲んで話しかけてもらえると助かります！"
        continue

    # 気持ちのゾーンーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーー
    found_mood = False

    for word in happy_words:
        if word in user_message.lower():
            mood = "happy"
            reply = random.choice(
                [
                    "元気そうで何よりです！何か良いことありましたか？",
                    "その調子ですね！今日も楽しく過ごしていきましょう！",
                ]
            )
            found_mood = True
            break

    if not found_mood:
        for word in tired_words:
            if word in user_message.lower():
                mood = "tired"
                reply = random.choice(
                    [
                        "お疲れ様です。あまり無理はなさらないでくださいね ☕",
                        "今日はゆっくり休みましょう。温かい飲み物でも飲んでリフレッシュしてくださいね。",
                    ]
                )
                found_mood = True
                break

    # ★ どの条件にも当てはまらなかった時（「あああ」など）
    if not found_mood:
        mood = "normal"
        reply = "すみません、上手く理解できませんでした。「『〇〇』について教えて」と聞くか、自己紹介を頼んでみてくださいね！"

┌──────────────────────────────────────────┐
  🤖✨ COIN：また、何時でも話しかけてくださいね～👋
└──────────────────────────────────────────┘


**C — Cute（かわいい）**
**O — Original（オリジナルの）**
**I — Innovative（革新的な）**
**N — なんて言ってるけど、元ネタがカネゴン**

Ai使用について、今回はif文という簡単な内容でしたが、似たような言葉を＝でひとくくりにし、２つを組み合わせるようにしたのでandを使おうとし、失敗しました。自分で書くことについて最初に言葉を定義した場合などの時は書けるのですが、コードの途中で言葉の定義が入った場合に理解が追い付いていないことが分かりました。API関連は前回の文をそのまま引用しました。wikiについては兼平　祥汰さんに影響を受けて使用しました。PCの時間を導入できることが分かったので「おはよう」に対してそれぞれの時間に合わせった内容を表示させることに成功しました。最後はelseの予定でしたがAIに聞いたところ前文のifの影響を受けるため間違っていると指摘されif notで修正しました。絵に関してはさすがに自分でしたい思いが勝ったのですが時間が足りず満足したものを書けなかったので顔以外動いてないことなど含め全体的にクオリティーの低さが目立っています。画像の導入に至っては自分１人では見当もつかずAIに聞きながら導入しました。本来に想定していた内容のポーカーやブラックジャックの導入や別windowでの表示、「もう一回」や「詳しく答えて」の対応、moodの他のパターンの絵が足りず間に合いませんでした。